# Patient extraction · top/bottom-decile transferred-XGBoost risk patients and their pathology reports

Data request (Elio). At each landmark (ADT +0 / +90 / +180 days) the platinum-trained federated XGBoost model, refit
**without Dana-Farber** and scored on the local DFCI ADT cohort in notebook 04, gives each patient a risk score. This
notebook takes the patients in the **top or bottom decile** of that score at **any** landmark and writes:

| File | Content |
|---|---|
| `patient_manifest.csv` | one row per patient: MRN, ADT start, overall `risk_group` (`top`, `bottom`, or `top and bottom`), risk score / rank percentile / top-or-bottom-decile flag at each landmark, AVPC / platinum / NEPC event flag and days from ADT start, number of pathology notes |
| `pathology_reports.tsv` | one row per pathology report (all dates): every manifest column plus the note metadata (`RPT_ID`, `EVENT_DATE`, `PROC_DESC`, `RPT_TYPE`, `FILE`, days from ADT start) and `RPT_TEXT` |
| `pathology_term_enrichment.csv` | terms overrepresented in the top / bottom decile's pre-landmark pathology reports vs the middle 80%, and in top vs bottom (both directions), at each landmark (aggregate counts only) |
| `pathology_topics.csv` | optional: BERTopic topics fit on the same pre-landmark reports (topic size and top words only; no report text) |
| `pathology_topic_enrichment.csv` | optional: topics overrepresented in the same contrasts (aggregate counts only) |
| `pathology_lexicon_enrichment.csv` | curated lexicon (neuroendocrine terms and markers, IHC, histology, Gleason) in the same contrasts (aggregate counts only) |
| `pathology_score_association.csv` | each term / lexicon feature vs the continuous risk percentile (Mann-Whitney, stratified van Elteren) |
| `pathology_text_classifier.csv` | cross-validated AUC of TF-IDF (and embedding) classifiers for each group pair vs a permutation null |

Output folder: `<COMPASS data root>/elio_data_requests/high_vs_low_risk_transfer_pxs/`. **The manifest and report files contain PHI**
(MRNs, report text); they stay under the protected data root and are never copied into the repository.

Run notebook 04 first: this reads its protected `all_models/transfer_scores.csv`. The risk score does not depend on the
evaluated endpoint, so the platinum rows are used. Deciles are taken among patients with finite numeric scores at that landmark.
Missing, nonnumeric, or infinite scores are excluded from cutoffs and ranks and labeled `not scored` in the manifest.
A patient is flagged `top` when their score is at or above the landmark's 90th percentile and `bottom` when at or below
its 10th. Ties at a cutoff are included, so groups may exceed 10%. There is no middle group: a landmark where the patient is in
neither decile has a blank `decile_group_lm<landmark>`. Extraction stops if a landmark has no valid scores
or its two cutoffs coincide, because that landmark cannot separate high- and low-risk groups.

## Configuration

In [ ]:
import os, sys
from pathlib import Path

sys.path.insert(0, "../../pipeline")
import compass_pipeline as cp

PROJECT_ROOT = cp.PROJECT_ROOT
for _p in (str(PROJECT_ROOT),):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import numpy as np
import pandas as pd

from survival_common import federated_endpoint_transfer as fet
from survival_common import federated_scoring as fs

DATA_ROOT = Path(cp._PROFILE_OUTPUT_ROOT)
RHINO_ROOT = Path(os.environ.get("COMPASS_RHINO_ROOT", str(DATA_ROOT / "rhino_data_v2")))

# >>> Edit <<<
SCORES_PATH = RHINO_ROOT / "federated_transfer_adt_no_DFCI" / "all_models" / "transfer_scores.csv"   # notebook 04
DATA_PATH   = DATA_ROOT / "longitudinal_prediction_data_adt.csv"                                      # ADT anchor + endpoints
PATHOLOGY_PATH = Path(os.environ.get(
    "PROFILE_PATHOLOGY_NOTES",
    "/data/gusev/USERS/jpconnor/data/PROFILE_DATA/CLINICAL_NOTES/PATHOLOGY_NOTES.parquet"))
OUTPUT_DIR = DATA_ROOT / "elio_data_requests" / "high_vs_low_risk_transfer_pxs"

MODEL_FAMILY = "xgboost_cox"
SCORE_ENDPOINT = "platinum"        # the model is platinum-trained; risk_score is identical across endpoints
LANDMARKS = (0, 90, 180)
DECILE = 0.10                      # top and bottom 10% at each landmark
ENDPOINTS = ("avpc", "platinum", "nepc")

# Pathology term enrichment (top / bottom decile vs middle 80%)
TERM_MIN_PATIENTS = 5              # a term must appear in >= this many patients at the landmark
TERM_NGRAMS = (1, 2)               # unigrams and bigrams
TERM_FDR = 0.05                    # Benjamini-Hochberg threshold for the displayed tables
TERM_TOP_N = 25                    # terms shown per landmark x group
TERM_NEGATION = True               # count negated words ("negative for synaptophysin") as separate neg_ terms
NEG_WINDOW = 6                     # content words a negation cue reaches forward (half that backward)

# Stratified sensitivity tests (Mantel-Haenszel, van Elteren): in-window report-count band x any non-prostate specimen
ADJUST_REPORT_BANDS = (1, 2, 4)    # band lower edges: 1, 2-3, 4+ reports
SPECIMEN_HEADER_CHARS = 300        # report characters (after PROC_DESC) searched for the specimen site

# Cross-validated text classifiers vs a permutation null
TEXT_C = 1.0                       # inverse regularization strength
TEXT_FOLDS = 5
TEXT_PERMUTATIONS = 100
TEXT_SEED = 42
N_JOBS = int(os.environ.get("SLURM_CPUS_PER_TASK", 1))

# Optional BERTopic on the same reports, from precomputed Clinical ModernBERT embeddings (read-only, from the
# clinical_text_embedding_project's knit_embeddings.py). The section skips if packages or files are missing.
CTEP_PROCESSED_DIR = Path(os.environ.get(
    "CTEP_DATA_PATH", "/data/gusev/USERS/jpconnor/data/clinical_text_embedding_project/"
)) / "batched_datasets" / "processed_datasets"
EMBED_METADATA_PATH = CTEP_PROCESSED_DIR / "full_clinical_notes_embeddings_metadata.parquet"
EMBED_ARRAY_PATH = CTEP_PROCESSED_DIR / "full_clinical_notes_embeddings_as_array.npy.zst"
TOPIC_MIN_SIZE = 30                # HDBSCAN min_cluster_size, in reports
TOPIC_SEED = 42                    # UMAP random_state

## Risk scores and deciles

One XGBoost full-model score per patient and landmark. The age-only baseline and any other family are dropped, and the
selection must leave exactly one row per patient and landmark.

In [ ]:
scores = pd.read_csv(SCORES_PATH, low_memory=False)
print(scores.groupby(["model_family", "config"]).size())

sel = scores[(scores["model_family"] == MODEL_FAMILY) & (scores["config"] != "baseline")
             & (scores["endpoint"] == SCORE_ENDPOINT) & (scores["landmark_days"].isin(LANDMARKS))]
sel = sel[[fs.ID_COL, "landmark_days", "risk_score"]]
dup = sel.duplicated([fs.ID_COL, "landmark_days"], keep=False)
assert not dup.any(), f"{dup.sum()} rows: more than one {MODEL_FAMILY} model/arm per patient and landmark; narrow the filter above"
assert set(sel["landmark_days"]) == set(LANDMARKS), f"landmarks found: {sorted(sel['landmark_days'].unique())}"

def decile_flags(g):
    if not 0 < DECILE < 0.5:
        raise ValueError("DECILE must be between 0 and 0.5 (exclusive).")
    g = g.copy()
    risk = pd.to_numeric(g["risk_score"], errors="coerce").astype(float)
    risk = risk.where(np.isfinite(risk))
    valid = risk.notna()
    landmark = g["landmark_days"].iloc[0]
    if not valid.any():
        raise ValueError(f"Landmark {landmark}: no finite numeric risk scores; extraction stopped.")
    lo, hi = risk[valid].quantile([DECILE, 1 - DECILE])
    if lo >= hi:
        raise ValueError(
            f"Landmark {landmark}: bottom and top cutoffs coincide; "
            "cannot separate high- and low-risk groups. Extraction stopped."
        )
    g["risk_score"] = risk
    g["risk_percentile"] = risk.rank(pct=True, method="average")
    # Only the two extreme deciles are labelled; everything in between stays blank.
    g["decile_group"] = pd.Series(pd.NA, index=g.index, dtype="string")
    g.loc[risk >= hi, "decile_group"] = "top"
    g.loc[risk <= lo, "decile_group"] = "bottom"
    g.loc[~valid, "decile_group"] = "not scored"
    g.attrs.update(lo=lo, hi=hi)
    return g

by_lm = {lm: decile_flags(g) for lm, g in sel.groupby("landmark_days")}
cutoffs = pd.DataFrame({lm: {"n_scored": int(g["risk_score"].notna().sum()),
                            "n_not_scored": int(g["risk_score"].isna().sum()),
                            "bottom_cutoff": g.attrs["lo"], "top_cutoff": g.attrs["hi"],
                            "n_top": int((g["decile_group"] == "top").sum()),
                            "n_bottom": int((g["decile_group"] == "bottom").sum())} for lm, g in by_lm.items()}).T
cutoffs

In [ ]:
wide = None
for lm, g in by_lm.items():
    part = g.rename(columns={"risk_score": f"risk_score_lm{lm}", "risk_percentile": f"risk_percentile_lm{lm}",
                             "decile_group": f"decile_group_lm{lm}"}).drop(columns="landmark_days")
    wide = part if wide is None else wide.merge(part, on=fs.ID_COL, how="outer")

group_cols = [f"decile_group_lm{lm}" for lm in LANDMARKS]
extreme = wide[group_cols].isin(["top", "bottom"]).any(axis=1)
cohort = wide[extreme].rename(columns={fs.ID_COL: "DFCI_MRN"}).copy()
for lm in LANDMARKS:
    col = f"decile_group_lm{lm}"
    cohort[f"top_decile_lm{lm}"] = cohort[col].eq("top").fillna(False).astype(bool)
    cohort[f"bottom_decile_lm{lm}"] = cohort[col].eq("bottom").fillna(False).astype(bool)
    # Absent from this landmark's score file -> not scored; scored but in neither decile stays blank.
    cohort.loc[cohort[f"risk_score_lm{lm}"].isna(), col] = "not scored"
is_top = cohort[[f"top_decile_lm{lm}" for lm in LANDMARKS]].any(axis=1)
is_bottom = cohort[[f"bottom_decile_lm{lm}" for lm in LANDMARKS]].any(axis=1)
cohort["risk_group"] = np.select([is_top & is_bottom, is_top], ["top and bottom", "top"], default="bottom")
print(f"{len(cohort):,} patients top/bottom decile at >=1 landmark (of {len(wide):,} patients in the score file)")
print(cohort["risk_group"].value_counts().to_string())

## ADT start and endpoints

ADT start is `TREATMENT_ANCHOR_DATE`, the anchor the landmarks are measured from. Each endpoint's date is used only when
its indicator is 1 (the date columns are back-filled for non-events upstream). `days_to_<endpoint>` counts from ADT start
and can be negative if the event was dated before ADT.

In [ ]:
def naive_dt(s):
    """Parse to datetime and drop any timezone (keeps wall-clock time) so note and ADT dates are comparable."""
    s = pd.to_datetime(s, errors="coerce", format="mixed", utc=False)
    if getattr(s.dt, "tz", None) is not None:
        s = s.dt.tz_localize(None)
    return s

local = fet.read_local_frame(DATA_PATH, ENDPOINTS)
cols = [fet.LOCAL_ID_COL, "TREATMENT_ANCHOR_DATE"] + [c for ep in ENDPOINTS for c in fet.ENDPOINT_SOURCES[ep]]
pt = local[cols].drop_duplicates(fet.LOCAL_ID_COL).copy()
pt["adt_start_datetime"] = naive_dt(pt["TREATMENT_ANCHOR_DATE"])
for ep in ENDPOINTS:
    ind, dt = fet.ENDPOINT_SOURCES[ep]
    event = pd.to_numeric(pt[ind], errors="coerce").fillna(0).eq(1)
    date = naive_dt(pt[dt]).where(event)
    pt[f"{ep}_event"] = event
    pt[f"{ep}_event_date"] = date
    pt[f"days_adt_to_{ep}"] = (date - pt["adt_start_datetime"]).dt.days
keep = [fet.LOCAL_ID_COL, "adt_start_datetime"] + [c for ep in ENDPOINTS for c in
        (f"{ep}_event", f"{ep}_event_date", f"days_adt_to_{ep}")]
pt = pt[keep].rename(columns={fet.LOCAL_ID_COL: "DFCI_MRN"})
pt["DFCI_MRN"] = pd.to_numeric(pt["DFCI_MRN"], errors="coerce").astype("Int64")
cohort["DFCI_MRN"] = pd.to_numeric(cohort["DFCI_MRN"], errors="coerce").astype("Int64")

manifest = cohort.merge(pt, on="DFCI_MRN", how="left", validate="one_to_one")
assert manifest["adt_start_datetime"].notna().all(), "scored patients missing an ADT start in the local frame"
manifest[[f"{ep}_event" for ep in ENDPOINTS]].sum()

## Pathology reports

Every pathology report on file for these patients, at any date (including after ADT start and after any event). Report
dates relative to ADT start are given so the request can be restricted downstream.

In [ ]:
import polars as pl

notes = (pl.scan_parquet(PATHOLOGY_PATH)
         .filter(pl.col("DFCI_MRN").cast(pl.Int64, strict=False).is_in(manifest["DFCI_MRN"].dropna().astype("int64").tolist()))
         .collect().to_pandas())
notes["DFCI_MRN"] = pd.to_numeric(notes["DFCI_MRN"], errors="coerce").astype("Int64")
notes["EVENT_DATE"] = naive_dt(notes["EVENT_DATE"])
print(f"{len(notes):,} pathology reports for {notes['DFCI_MRN'].nunique():,} of {len(manifest):,} patients")

counts = notes.groupby("DFCI_MRN").size().rename("n_pathology_notes")
manifest = manifest.merge(counts, on="DFCI_MRN", how="left")
manifest["n_pathology_notes"] = manifest["n_pathology_notes"].fillna(0).astype(int)
manifest = manifest.sort_values("DFCI_MRN").reset_index(drop=True)

# Column order: identity, ADT start, overall top/bottom group, per-landmark score / percentile / flags, endpoints, note count.
lm_cols = [c for lm in LANDMARKS for c in (f"risk_score_lm{lm}", f"risk_percentile_lm{lm}", f"decile_group_lm{lm}",
                                          f"top_decile_lm{lm}", f"bottom_decile_lm{lm}")]
ep_cols = [c for ep in ENDPOINTS for c in (f"{ep}_event", f"days_adt_to_{ep}", f"{ep}_event_date")]
manifest = manifest[["DFCI_MRN", "adt_start_datetime", "risk_group"] + lm_cols + ep_cols + ["n_pathology_notes"]]
print(f"Manifest ready: {len(manifest):,} patients; "
      f"{manifest['n_pathology_notes'].eq(0).sum():,} without pathology reports")

## Write

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
manifest_path = OUTPUT_DIR / "patient_manifest.csv"
manifest.to_csv(manifest_path, index=False)

note_cols = [c for c in ("RPT_ID", "EVENT_DATE", "PROC_DESC", "RPT_TYPE", "FILE") if c in notes.columns]
rep = notes[["DFCI_MRN"] + note_cols + ["RPT_TEXT"]].rename(columns={
    "EVENT_DATE": "note_date", **{c: f"note_{c.lower()}" for c in note_cols if c != "EVENT_DATE"}})
rep = manifest.merge(rep, on="DFCI_MRN", how="inner", validate="one_to_many")
rep["note_days_from_adt_start"] = (rep["note_date"] - rep["adt_start_datetime"]).dt.days
text_col = rep.pop("RPT_TEXT").rename("note_text")
rep = pd.concat([rep, text_col], axis=1).sort_values(["DFCI_MRN", "note_date"], kind="stable")
# Report bodies carry tabs and newlines; pandas quotes such fields, which any TSV reader honours.
tsv_path = OUTPUT_DIR / "pathology_reports.tsv"
rep.to_csv(tsv_path, sep="\t", index=False)

check = pd.read_csv(tsv_path, sep="\t", low_memory=False)
assert len(check) == len(rep) == int(manifest["n_pathology_notes"].sum())
print(f"wrote {manifest_path} ({len(manifest):,} patients)")
print(f"wrote {tsv_path} ({len(rep):,} reports; round-trip row count verified)")

## Pathology term enrichment: top / bottom decile vs middle 80%, and top vs bottom

At each landmark, patients fall into three groups by their score at that landmark: **top** decile, **bottom** decile, and
**middle** (scored, in neither decile; ties at a cutoff stay in the extreme groups, so this is at most 80%). Four
contrasts per landmark, each reading "overrepresented in `group` relative to `control`": top vs middle, bottom vs middle,
top vs bottom and bottom vs top. Top vs bottom is run in both directions because the test is one-sided. Each patient's
group can differ between landmarks.

- **Reports:** every scored patient's pathology reports dated on or before ADT start + landmark days, i.e. what was on
  file when the patient was scored. Later reports (including post-progression pathology) and undated reports are left out.
  Patients with no report in the window drop out of that landmark's denominators.
- **Terms:** lowercased letter-only unigrams and bigrams (numbers, dates and IDs drop out), English stop words removed,
  counted within each report so bigrams never span two reports. A term is kept if it appears in at least
  `TERM_MIN_PATIENTS` patients at the landmark, which also keeps patient-specific strings such as names out. With
  `TERM_NEGATION`, words in a negation scope are prefixed `neg_` (`neg_synaptophysin`) and counted separately
  (see the sensitivity section below).
- **Test:** a patient "has" a term if any of their in-window reports contains it. For each contrast, one-sided Fisher's
  exact test for overrepresentation in `group`, Benjamini-Hochberg q-values within each contrast, and the odds ratio
  with a +0.5 correction. Mean patient-level TF-IDF (sublinear term frequency over each patient's pooled counts) is reported
  for each group as a descriptive column. The `*_mh` columns repeat the test stratified by report count and specimen
  site (Mantel-Haenszel; see below).

In [ ]:
import bisect
import re

import polars as pl
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

TERM_GROUPS = ("top", "bottom")
CONTROL_GROUP = "middle"
# (group, control): one-sided tests for terms / topics overrepresented in group relative to control.
CONTRASTS = (("top", CONTROL_GROUP), ("bottom", CONTROL_GROUP), ("top", "bottom"), ("bottom", "top"))

# --- Negation scopes (NegEx-style), on lowercased text. The sentence-end rule is copied (not imported) from
# LLM_clinical_annotations/tasks/nepc/ne_gate.py (_SENTENCE_END; NE_GATE_VERSION "ne-gate-v3", commit
# edb1713ef454f6ece313fa675b5f9438937f831e), and the cue lists follow that file's negation cues.
NEG_SENTENCE_END = r"(?<!\bvs)(?<!\bdr)(?<!\bno)[.;!?](?=\s|$)"
NEG_PRE = re.compile(      # negate the next NEG_WINDOW content words
    r"\bno\b(?!\s*[.:#\d])|\b(?:not|without|neither|nor|negative\s+(?:for|in)|absence\s+of|free\s+of"
    r"|lack(?:s|ing)?(?:\s+of)?|rule\s+out|r/o)\b")
NEG_POST = re.compile(     # negate the previous NEG_WINDOW // 2 content words
    r"\b(?:not\s+(?:identified|seen|present|detected|appreciated|demonstrated)|negative(?!\s+(?:for|in)\b)"
    r"|absent|ruled\s+out|excluded)\b|\(\s*-\s*\)")
NEG_PSEUDO = re.compile(   # look like negation but are not
    r"\b(?:cannot|can\s*not|can't)\s+(?:be\s+)?(?:rule[ds]?\s+out|exclude[ds]?)\b"
    r"|\bnot\s+(?:be\s+)?(?:ruled\s+out|excluded)\b|\bno\s+longer\b|\bnot\s+only\b")
NEG_BREAK = re.compile(NEG_SENTENCE_END + r"|\n|\b(?:but|however|although|whereas|except|positive|positivity)\b")
# Words that complete a negation without carrying content ("no carcinoma identified"): never prefixed, not counted.
NEG_NEUTRAL = frozenset("identified seen present detected appreciated demonstrated noted observed found evidence "
                        "definite definitive convincing".split())
WORD_RE = re.compile(r"\b[a-z][a-z0-9]*\b")


def _overlaps(span, spans):
    return any(s < span[1] and span[0] < e for s, e in spans)


def negated_spans(text):
    """(start, end) of each content word inside a negation scope, in order; `text` must already be lowercased.

    A pre-negation cue ("no", "negative for") reaches forward NEG_WINDOW content words; a post-negation cue
    ("negative", "not identified") reaches back NEG_WINDOW // 2. Neither crosses a sentence end, a line break,
    "but" / "however" / ... or "positive". Post cues also stop at a comma, so "synaptophysin positive, chromogranin
    negative" negates chromogranin only, while "negative for synaptophysin, chromogranin and cd56" negates all three.
    An explicitly positive clause after a comma or conjunction ends a forward scope before its subject
    ("negative for synaptophysin, chromogranin positive"). Negative marker lists still span commas.
    Stop words and NEG_NEUTRAL words are skipped and not counted.
    """
    pseudo = [m.span() for m in NEG_PSEUDO.finditer(text)]
    post = [m.span() for m in NEG_POST.finditer(text) if not _overlaps(m.span(), pseudo)]
    pre = [m.span() for m in NEG_PRE.finditer(text) if not _overlaps(m.span(), pseudo + post)]
    if not (pre or post):
        return []
    cues = pseudo + pre + post
    words = [m.span() for m in WORD_RE.finditer(text)]
    starts = [s for s, _ in words]
    hard = [m.start() for m in NEG_BREAK.finditer(text)]
    soft = sorted(hard + [i for i, ch in enumerate(text) if ch == ","])

    def content(i):
        word = text[words[i][0]:words[i][1]]
        return word not in ENGLISH_STOP_WORDS and word not in NEG_NEUTRAL and not _overlaps(words[i], cues)

    marked = set()
    for _, end in pre:
        k = bisect.bisect_left(hard, end)
        stop = hard[k] if k < len(hard) else len(text)
        if re.match(r"positive\b|positivity\b", text[stop:]):
            # A trailing positive result applies to the subject before it, not just to subsequent words.
            # Prefer a comma so "..., chromogranin and cd56 positive" protects the entire positive list.
            left = max([end] + [e for s, e in post if end <= s and e <= stop])
            separators = list(re.finditer(r",", text[left:stop]))
            if not separators:
                separators = list(re.finditer(r"\b(?:and|or)\b", text[left:stop]))
            if separators:
                stop = left + separators[-1].end()
        i, n = bisect.bisect_left(starts, end), 0
        while i < len(words) and words[i][0] < stop and n < NEG_WINDOW:
            if content(i):
                marked.add(i)
                n += 1
            i += 1
    for start, _ in post:
        k = bisect.bisect_left(soft, start)
        stop = soft[k - 1] if k else -1
        i, n = bisect.bisect_left(starts, start) - 1, 0
        while i >= 0 and words[i][0] > stop and n < max(1, NEG_WINDOW // 2):
            if content(i):
                marked.add(i)
                n += 1
            i -= 1
    return [words[i] for i in sorted(marked)]


def mark_negation(text):
    """Lowercased text with each negated content word prefixed by neg_ (digits and punctuation untouched)."""
    text = text.lower()
    out, last = [], 0
    for start, _ in negated_spans(text):
        out += [text[last:start], "neg_"]
        last = start
    return "".join(out) + text[last:]


# --- Specimen site (for the stratified sensitivity test): the site named first in PROC_DESC + the report's opening.
SPECIMEN_SITES = (
    ("prostate", re.compile(r"\b(?:prostat\w*|turp|transurethral|seminal\s+vesicles?)\b")),
    ("non-prostate", re.compile(r"\b(?:bone|marrow|liver|hepatic|lymph|nodes?|nodal|lung|pulmonary|pleura\w*|adrenal"
                                r"|brain|dura\w*|spine|spinal|vertebra\w*|soft\s+tissue|peritone\w*|omentum|omental)\b")),
)


def specimen_site(header):
    """'prostate', 'non-prostate' or 'unspecified': whichever site is named first in a lowercased header."""
    hits = [(m.start(), site) for site, rx in SPECIMEN_SITES for m in [rx.search(header)] if m]
    return min(hits)[1] if hits else "unspecified"


# Per-landmark group for every scored patient; a blank decile group means scored but in neither decile.
groups_by_lm = {}
for lm, g in by_lm.items():
    ids = pd.Index(pd.to_numeric(g[fs.ID_COL], errors="coerce").astype("Int64"), name="DFCI_MRN")
    grp = pd.Series(g["decile_group"].fillna(CONTROL_GROUP).to_numpy(), index=ids, name="group")
    groups_by_lm[lm] = grp[grp.ne("not scored")]

scored_ids = sorted({int(i) for grp in groups_by_lm.values() for i in grp.index.dropna()})
# RPT_ID (when present) links reports to the optional embeddings; PROC_DESC (when present) helps type the specimen.
note_cols = ["DFCI_MRN", "EVENT_DATE", "RPT_TEXT"]
schema = pl.scan_parquet(PATHOLOGY_PATH).collect_schema().names()
note_cols += [c for c in ("RPT_ID", "PROC_DESC") if c in schema]
all_notes = (pl.scan_parquet(PATHOLOGY_PATH)
             .select(note_cols)
             .filter(pl.col("DFCI_MRN").cast(pl.Int64, strict=False).is_in(scored_ids))
             .collect().to_pandas())
all_notes["DFCI_MRN"] = pd.to_numeric(all_notes["DFCI_MRN"], errors="coerce").astype("Int64")
all_notes["note_day"] = naive_dt(all_notes["EVENT_DATE"]).dt.normalize()
all_notes["RPT_TEXT"] = all_notes["RPT_TEXT"].fillna("").astype(str)
all_notes = all_notes.merge(pt[["DFCI_MRN", "adt_start_datetime"]], on="DFCI_MRN", how="left", validate="many_to_one")
all_notes["note_row"] = np.arange(len(all_notes))      # row position, used to look up per-report lexicon hits
adt_day = all_notes["adt_start_datetime"].dt.normalize()

# Negation marking (TERM_TEXT) and specimen site, once per report inside the widest window.
widest = all_notes["note_day"].le(adt_day + np.timedelta64(int(max(LANDMARKS)), "D"))
all_notes["TERM_TEXT"] = ""
all_notes["specimen_site"] = "unspecified"
if widest.any():
    texts = all_notes.loc[widest, "RPT_TEXT"]
    all_notes.loc[widest, "TERM_TEXT"] = texts.map(mark_negation if TERM_NEGATION else str.lower).to_numpy()
    header = texts.str[:SPECIMEN_HEADER_CHARS]
    if "PROC_DESC" in all_notes:
        header = all_notes.loc[widest, "PROC_DESC"].fillna("").astype(str) + " " + header
    all_notes.loc[widest, "specimen_site"] = header.str.lower().map(specimen_site).to_numpy()

# Reports on or before ADT start + landmark; NaT dates (undated report or no ADT start) compare False and drop out.
# Each patient's stratum: in-window report count band (ADJUST_REPORT_BANDS) x any non-prostate-site report.
reports_by_lm, strata_by_lm, size_rows = {}, {}, []
for lm, grp in groups_by_lm.items():
    in_window = (all_notes["note_day"].le(adt_day + np.timedelta64(int(lm), "D"))
                 & all_notes["DFCI_MRN"].isin(grp.index))
    rep_lm = all_notes.loc[in_window, ["DFCI_MRN", "RPT_TEXT", "TERM_TEXT", "specimen_site", "note_row"]
                           + note_cols[3:]].reset_index(drop=True)
    reports_by_lm[lm] = rep_lm
    per = (rep_lm.assign(non_prostate=rep_lm["specimen_site"].eq("non-prostate"))
           .groupby("DFCI_MRN").agg(reports=("non_prostate", "size"), non_prostate=("non_prostate", "any")))
    band = np.searchsorted(ADJUST_REPORT_BANDS, per["reports"].to_numpy(), side="right") - 1
    strata_by_lm[lm] = pd.Series(band * 2 + per["non_prostate"].to_numpy(dtype=int), index=per.index, name="stratum")
    has_report = grp.index.isin(rep_lm["DFCI_MRN"])
    for name in (*TERM_GROUPS, CONTROL_GROUP):
        in_group = grp.eq(name).to_numpy()
        members = per.index.isin(grp.index[in_group])
        size_rows.append({"landmark": lm, "group": name, "patients_scored": int(in_group.sum()),
                          "patients_with_report": int((in_group & has_report).sum()),
                          "reports": int(rep_lm["DFCI_MRN"].isin(grp.index[in_group]).sum()),
                          "median_reports": float(per.loc[members, "reports"].median()) if members.any() else np.nan,
                          "patients_with_non_prostate_report": int(per.loc[members, "non_prostate"].sum())})

print(f"{all_notes['note_day'].isna().sum():,} undated pathology reports excluded; "
      f"{all_notes['adt_start_datetime'].isna().sum():,} reports from patients without an ADT start excluded")
sites = all_notes.loc[widest, "specimen_site"].value_counts()
print(f"Specimen site of reports on or before ADT start + {max(LANDMARKS)} days: "
      + ", ".join(f"{site} {int(sites.get(site, 0)):,}" for site in ("prostate", "non-prostate", "unspecified")))
group_sizes = pd.DataFrame(size_rows).set_index(["landmark", "group"])
group_sizes


In [ ]:
from scipy import sparse
from scipy.stats import hypergeom, norm
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer

# Applied after lowercasing: letters only, >= 3 characters, optionally negation-marked (neg_synaptophysin).
TOKEN_PATTERN = r"(?u)\b(?:neg_)?[a-z][a-z]{2,}\b"
MH_COLS = ["odds_ratio_mh", "p_value_mh", "q_value_mh"]
ENRICHMENT_COLS = ["landmark", "group", "control", "term", "n_group", "n_control", "group_with_term", "control_with_term",
                   "pct_group", "pct_control", "odds_ratio", "p_value", "q_value", *MH_COLS,
                   "mean_tfidf_group", "mean_tfidf_control"]


def bh_qvalues(p):
    """Benjamini-Hochberg q-values over the finite p values; NaN stays NaN (cluster scipy predates
    false_discovery_control)."""
    p = np.asarray(p, dtype=float)
    q = np.full(p.shape, np.nan)
    ok = np.flatnonzero(np.isfinite(p))
    if ok.size:
        order = ok[np.argsort(p[ok])]
        ranked = p[order] * ok.size / np.arange(1, ok.size + 1)
        q[order] = np.minimum(np.minimum.accumulate(ranked[::-1])[::-1], 1.0)
    return q


def patient_term_matrices(reports, ids):
    """Patient x term presence and TF-IDF, with terms counted within each report and summed per patient.

    Uses the negation-marked TERM_TEXT when the reports carry it, else RPT_TEXT.
    """
    text = reports["TERM_TEXT" if "TERM_TEXT" in reports else "RPT_TEXT"].to_numpy()
    vec = CountVectorizer(lowercase=True, stop_words="english", token_pattern=TOKEN_PATTERN,
                          ngram_range=TERM_NGRAMS, min_df=TERM_MIN_PATIENTS)
    try:
        # Report-level min_df is a safe prefilter: a term in >= k patients is in >= k reports.
        counts = vec.fit_transform(text)
    except ValueError:  # no term survives (or fewer reports than TERM_MIN_PATIENTS)
        return np.array([], dtype=object), None, None
    rows = ids.get_indexer(reports["DFCI_MRN"])
    to_patient = sparse.csr_matrix((np.ones(len(rows)), (rows, np.arange(len(rows)))),
                                   shape=(len(ids), len(rows)))
    patient_counts = (to_patient @ counts).tocsc()
    present = (patient_counts > 0).astype(np.int64)
    keep = np.flatnonzero(np.asarray(present.sum(axis=0)).ravel() >= TERM_MIN_PATIENTS)
    if not keep.size:
        return np.array([], dtype=object), None, None
    patient_counts, present = patient_counts[:, keep], present[:, keep]
    tfidf = TfidfTransformer(sublinear_tf=True).fit_transform(patient_counts)
    return vec.get_feature_names_out()[keep], present.tocsr(), tfidf.tocsr()


def colsum(m):
    return np.asarray(m.sum(axis=0)).ravel().astype(float)


def mantel_haenszel(present, in_group, control, strata):
    """One-sided, continuity-corrected Mantel-Haenszel test and MH odds ratio per feature column, across strata.

    A stratum without patients from both sides carries no information and is skipped; if none is left, both are NaN.
    The odds ratio adds 0.5 to every cell of a stratum that has an empty cell.
    """
    n_feat = present.shape[1]
    A, E, V, R, S = (np.zeros(n_feat) for _ in range(5))
    informative = False
    for k in pd.unique(strata[in_group | control]):
        g, c = in_group & (strata == k), control & (strata == k)
        n1, n0 = int(g.sum()), int(c.sum())
        if not (n1 and n0):
            continue
        informative = True
        n = n1 + n0
        a, cc = colsum(present[g]), colsum(present[c])
        b, d, m = n1 - a, n0 - cc, a + cc
        A += a
        E += n1 * m / n
        V += n1 * n0 * m * (n - m) / (n ** 2 * (n - 1))
        h = 0.5 * ((a == 0) | (b == 0) | (cc == 0) | (d == 0))
        R += (a + h) * (d + h) / (n + 4 * h)
        S += (b + h) * (cc + h) / (n + 4 * h)
    if not informative:
        return np.full(n_feat, np.nan), np.full(n_feat, np.nan)
    with np.errstate(divide="ignore", invalid="ignore"):
        p = np.where(V > 0, norm.sf((A - E - 0.5) / np.sqrt(V)), 1.0)
    return R / S, p


def fisher_enrichment(present, label, features, landmark, strata=None):
    """One-sided Fisher overrepresentation of each feature column for each (group, control) in CONTRASTS.

    present: patients x features 0/1 sparse matrix; label: each patient's group; strata (optional): each patient's
    stratum for the Mantel-Haenszel columns (NaN without it). Rows come out in CONTRASTS order, len(features) rows
    per contrast whose group and control both have patients.
    """
    out = []
    for name, ref in CONTRASTS:
        in_group, control = label == name, label == ref
        n1, n0 = int(in_group.sum()), int(control.sum())
        if n1 == 0 or n0 == 0:
            continue
        a, c = colsum(present[in_group]), colsum(present[control])
        p = hypergeom.sf(a - 1, n1 + n0, a + c, n1)       # P(X >= a): one-sided Fisher
        if strata is None:
            or_mh = p_mh = np.full(len(features), np.nan)
        else:
            or_mh, p_mh = mantel_haenszel(present, in_group, control, np.asarray(strata))
        out.append(pd.DataFrame({
            "landmark": landmark, "group": name, "control": ref, "feature": features, "n_group": n1, "n_control": n0,
            "group_with_feature": a.astype(int), "control_with_feature": c.astype(int),
            "pct_group": a / n1, "pct_control": c / n0,
            "odds_ratio": ((a + 0.5) * (n0 - c + 0.5)) / ((n1 - a + 0.5) * (c + 0.5)),
            "p_value": p, "q_value": bh_qvalues(p),
            "odds_ratio_mh": or_mh, "p_value_mh": p_mh, "q_value_mh": bh_qvalues(p_mh),
        }))
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame()


def sort_contrasts(df):
    """Landmark, then CONTRASTS order, then p value and odds ratio."""
    rank = {pair: i for i, pair in enumerate(CONTRASTS)}
    order = pd.Series([rank[pair] for pair in zip(df["group"], df["control"])], index=df.index, dtype=float)
    return (df.assign(_contrast=order)
            .sort_values(["landmark", "_contrast", "p_value", "odds_ratio"], ascending=[True, True, True, False])
            .drop(columns="_contrast").reset_index(drop=True))


def contrast_title(lm, name, ref):
    return f"Landmark {lm} · {name} decile vs {'middle' if ref == CONTROL_GROUP else ref + ' decile'}"


term_matrices = {}   # landmark -> (patient ids, terms, presence, TF-IDF); reused by the score association cell


def term_enrichment(reports, groups, landmark):
    """Fisher overrepresentation of each term for each contrast in CONTRASTS at one landmark."""
    ids = pd.Index(sorted(reports["DFCI_MRN"].unique()), dtype="Int64")   # patients with >= 1 in-window report
    terms, present, tfidf = patient_term_matrices(reports, ids)
    term_matrices[landmark] = (ids, terms, present, tfidf)
    if not len(terms):
        return pd.DataFrame(columns=ENRICHMENT_COLS)
    label = groups.reindex(ids).to_numpy()
    res = fisher_enrichment(present, label, terms, landmark, strata_by_lm[landmark].reindex(ids).to_numpy())
    if res.empty:
        return pd.DataFrame(columns=ENRICHMENT_COLS)
    mean_tfidf = {name: np.asarray(tfidf[label == name].mean(axis=0)).ravel()
                  for name in set(res["group"]) | set(res["control"])}
    pairs = list(res[["group", "control"]].drop_duplicates().itertuples(index=False))
    res["mean_tfidf_group"] = np.concatenate([mean_tfidf[g] for g, _ in pairs])
    res["mean_tfidf_control"] = np.concatenate([mean_tfidf[c] for _, c in pairs])
    return res.rename(columns={"feature": "term", "group_with_feature": "group_with_term",
                               "control_with_feature": "control_with_term"})


enrichment = pd.concat(
    [term_enrichment(reports_by_lm[lm], groups_by_lm[lm], lm) for lm in LANDMARKS if lm in reports_by_lm],
    ignore_index=True,
).reindex(columns=ENRICHMENT_COLS)
enrichment = sort_contrasts(enrichment)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
enrichment_path = OUTPUT_DIR / "pathology_term_enrichment.csv"
enrichment.to_csv(enrichment_path, index=False)
print(f"wrote {enrichment_path} ({len(enrichment):,} term x contrast rows)")


In [ ]:
try:
    from IPython.display import display
except ImportError:  # plain-Python runs
    display = print

show_cols = ["term", "group_with_term", "n_group", "pct_group", "control_with_term", "n_control", "pct_control",
             "odds_ratio", "q_value", "odds_ratio_mh", "q_value_mh", "mean_tfidf_group", "mean_tfidf_control"]
for lm in LANDMARKS:
    for name, ref in CONTRASTS:
        sub = enrichment[(enrichment["landmark"] == lm) & (enrichment["group"] == name) & (enrichment["control"] == ref)]
        title = contrast_title(lm, name, ref)
        if sub.empty:
            print(f"\n{title}: no terms (too few patients or reports)")
            continue
        sig = sub[sub["q_value"] < TERM_FDR]
        note = f"q < {TERM_FDR}: {len(sig):,} terms"
        if sig.empty:
            sig, note = sub, f"none at q < {TERM_FDR}; top {TERM_TOP_N} by p shown"
        top = sig.sort_values(["p_value", "odds_ratio"], ascending=[True, False]).head(TERM_TOP_N)
        print(f"\n{title} ({note})")
        display(top[show_cols].round(3).reset_index(drop=True))

## Optional: BERTopic topics in the same reports

Groups, reports and the date window are the same as above. Instead of single terms, reports are clustered into topics,
and topics are tested for overrepresentation in the same four contrasts.

- **Embeddings:** precomputed, mean-pooled Clinical ModernBERT embeddings of each pathology report, read (never written)
  from the clinical_text_embedding_project (`full_clinical_notes_embeddings_as_array.npy.zst` and its metadata parquet),
  joined on `RPT_ID`. No language model is loaded and nothing is downloaded. Reports outside that project's cohort have
  no embedding; the coverage table shows how many patients each contrast keeps.
- **Model:** one BERTopic fit on the union of in-window, embedded reports across landmarks (each report once), so topic
  numbers mean the same thing at every landmark. UMAP (5 dimensions, cosine, `TOPIC_SEED`) then HDBSCAN
  (`TOPIC_MIN_SIZE` reports). Topic words use only the terms that appear in at least `TERM_MIN_PATIENTS` patients, with the same
  `neg_` marking.
- **Test:** a patient "has" a topic if any of their in-window embedded reports is assigned to it. Outlier reports
  (topic -1) are not a feature, but their patients stay in the denominators. Denominators are patients with at least one
  embedded in-window report. Topics in fewer than `TERM_MIN_PATIENTS` patients at a landmark are dropped there.
  Same one-sided Fisher test, BH q-values and odds ratio as the terms.
- **PHI:** BERTopic keeps representative documents (raw report text) inside the fitted model. They are dropped from
  every table and the model is never saved; only topic sizes and top words are written.

The section needs `bertopic`, `umap-learn`, `hdbscan` and `zstandard` (commented out in `requirements.txt`). Without
them, or without the embedding files, it prints why it skipped and writes nothing; the term enrichment above is unaffected.
Loading the embeddings needs only `zstandard`; they are reused by the embedding classifier below even when BERTopic is
not installed.

In [ ]:
import importlib
import io

TOPIC_PACKAGES = {"zstandard": "zstandard", "bertopic": "bertopic", "umap": "umap-learn", "hdbscan": "hdbscan"}


def optional_import(name):
    try:
        return importlib.import_module(name)
    except Exception:   # ImportError, or a broken install (e.g. numba) that fails on import
        return None


def rpt_key(values):
    """RPT_ID as a string join key; whole-number IDs lose any float formatting ("12.0" -> "12")."""
    s = pd.Series(values).reset_index(drop=True)
    num = pd.to_numeric(s, errors="coerce")
    whole = num.notna() & np.isfinite(num) & (num % 1 == 0)
    key = s.astype("string").str.strip()
    key[whole] = num[whole].astype("int64").astype(str)
    return key.where(s.notna())


def _read_exact(fh, n):
    parts, got = [], 0
    while got < n:
        chunk = fh.read(n - got)
        if not chunk:
            raise ValueError("embedding array ended before its header's shape")
        parts.append(chunk)
        got += len(chunk)
    return b"".join(parts)


def read_embedding_rows(path, rows, zstd, chunk_rows=16384):
    """Rows `rows` (sorted, unique) of a zstd-compressed 2-D .npy array, as float32, in one streaming pass.

    The full array (every note type for the embedding cohort) is never held in memory: it is decompressed in
    chunk_rows blocks and only the requested rows are kept.
    """
    rows = np.asarray(rows, dtype=np.int64)
    with open(path, "rb") as fh, zstd.ZstdDecompressor().stream_reader(fh) as raw:
        version = np.lib.format.read_magic(raw)
        readers = {(1, 0): np.lib.format.read_array_header_1_0, (2, 0): np.lib.format.read_array_header_2_0}
        if version not in readers:
            raise ValueError(f"unsupported .npy version {version}")
        shape, fortran_order, dtype = readers[version](raw)
        if fortran_order or len(shape) != 2 or dtype.hasobject:
            raise ValueError(f"expected a C-order 2-D numeric array, got shape {shape}")
        n_rows, dim = shape
        if rows.size and (rows[0] < 0 or rows[-1] >= n_rows):
            raise ValueError(f"EMBEDDING_INDEX outside the array's {n_rows:,} rows")
        out = np.empty((rows.size, dim), dtype=np.float32)
        start, stop = 0, int(rows[-1]) + 1 if rows.size else 0
        while start < stop:
            k = min(chunk_rows, stop - start)
            block = np.frombuffer(_read_exact(raw, k * dim * dtype.itemsize), dtype=dtype).reshape(k, dim)
            lo, hi = np.searchsorted(rows, [start, start + k])
            out[lo:hi] = block[rows[lo:hi] - start]
            start += k
    return out


topic_mods = {name: optional_import(name) for name in TOPIC_PACKAGES}
missing = [pip for name, pip in TOPIC_PACKAGES.items() if topic_mods[name] is None]

# Loading the embeddings needs only zstandard; the text classifier at the end of the notebook reuses them.
embed_skip = None
if topic_mods["zstandard"] is None:
    embed_skip = "packages not installed: zstandard"
elif not (EMBED_METADATA_PATH.exists() and EMBED_ARRAY_PATH.exists()):
    embed_skip = f"precomputed embeddings not found under {EMBED_ARRAY_PATH.parent}"
elif "RPT_ID" not in all_notes:
    embed_skip = "pathology notes have no RPT_ID to join the embeddings on"

embed_docs = report_embeddings = None   # embed_docs row i <-> report_embeddings row i
if embed_skip is None:
    # Each in-window report once, across landmarks (the windows only differ in which reports they include).
    embed_docs = (pd.concat(reports_by_lm.values(), ignore_index=True)
                  .assign(rpt_key=lambda d: rpt_key(d["RPT_ID"]).to_numpy())
                  .dropna(subset=["rpt_key"]).drop_duplicates("rpt_key", ignore_index=True))
    meta = (pl.scan_parquet(EMBED_METADATA_PATH)
            .filter(pl.col("NOTE_TYPE") == "Pathology")
            .select("RPT_ID", "EMBEDDING_INDEX")
            .collect().to_pandas())
    meta["rpt_key"] = rpt_key(meta["RPT_ID"]).to_numpy()
    meta = meta.dropna(subset=["rpt_key", "EMBEDDING_INDEX"]).drop_duplicates("rpt_key")
    n_window = len(embed_docs)
    embed_docs = (embed_docs.merge(meta[["rpt_key", "EMBEDDING_INDEX"]], on="rpt_key", how="inner",
                                   validate="one_to_one")
                  .sort_values("EMBEDDING_INDEX", ignore_index=True))
    print(f"{len(embed_docs):,} of {n_window:,} in-window pathology reports have a precomputed embedding")
    if embed_docs.empty:
        embed_skip, embed_docs = "no in-window pathology report has a precomputed embedding", None
    else:
        report_embeddings = read_embedding_rows(EMBED_ARRAY_PATH, embed_docs["EMBEDDING_INDEX"].astype("int64").to_numpy(),
                                                topic_mods["zstandard"])

topic_skip = None
if missing:
    topic_skip = f"packages not installed: {', '.join(missing)}"
elif embed_skip is not None:
    topic_skip = embed_skip
elif len(embed_docs) <= 2 * TOPIC_MIN_SIZE:
    topic_skip = f"only {len(embed_docs):,} embedded reports (need more than {2 * TOPIC_MIN_SIZE})"
else:
    # Topic words come only from terms in >= TERM_MIN_PATIENTS patients, the same rule as the term enrichment.
    vocab, _, _ = patient_term_matrices(embed_docs, pd.Index(sorted(embed_docs["DFCI_MRN"].unique()), dtype="Int64"))
    if not len(vocab):
        topic_skip = f"no term appears in >= {TERM_MIN_PATIENTS} patients' embedded reports"

topic_docs = None
if topic_skip is None:
    topic_model = topic_mods["bertopic"].BERTopic(
        embedding_model=None,   # embeddings are passed in: no backend is selected or downloaded
        umap_model=topic_mods["umap"].UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine",
                                           random_state=TOPIC_SEED),
        hdbscan_model=topic_mods["hdbscan"].HDBSCAN(min_cluster_size=TOPIC_MIN_SIZE, metric="euclidean",
                                                    cluster_selection_method="eom", prediction_data=False),
        vectorizer_model=CountVectorizer(vocabulary=list(vocab), ngram_range=TERM_NGRAMS, lowercase=True,
                                         token_pattern=TOKEN_PATTERN),
        calculate_probabilities=False,
        verbose=False,
    )
    # Documents are the negation-marked text, so topic words use the same vocabulary as the term enrichment.
    assigned, _ = topic_model.fit_transform(embed_docs["TERM_TEXT"].tolist(), embeddings=report_embeddings)
    topic_docs = embed_docs.assign(topic=np.asarray(assigned, dtype=int))
    info = topic_model.get_topic_info()
    words = {int(t): " ".join(w for w, _ in (topic_model.get_topic(t) or [])) for t in info["Topic"]}
    del topic_model, info   # the model holds representative report text: never saved or shown
    topic_table = (topic_docs.groupby("topic")
                   .agg(n_reports=("rpt_key", "size"), n_patients=("DFCI_MRN", "nunique"))
                   .reset_index())
    topic_table["top_words"] = topic_table["topic"].map(words).fillna("")
    topic_table.loc[topic_table["topic"].eq(-1), "top_words"] = "(outliers: no topic)"
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    topics_path = OUTPUT_DIR / "pathology_topics.csv"
    topic_table.to_csv(topics_path, index=False)
    print(f"wrote {topics_path} ({topic_table['topic'].ge(0).sum():,} topics, "
          f"{topic_table.loc[topic_table['topic'].eq(-1), 'n_reports'].sum():,} outlier reports)")
else:
    print(f"BERTopic section skipped ({topic_skip}); no topic files written.")


In [ ]:
TOPIC_COLS = ["landmark", "group", "control", "topic", "top_words", "n_group", "n_control", "group_with_topic",
              "control_with_topic", "pct_group", "pct_control", "odds_ratio", "p_value", "q_value", *MH_COLS]


def topic_enrichment(docs, groups, landmark):
    """Fisher overrepresentation of each topic for each contrast in CONTRASTS at one landmark."""
    ids = pd.Index(sorted(docs["DFCI_MRN"].unique()), dtype="Int64")   # patients with >= 1 embedded report
    topics = np.array(sorted(t for t in docs["topic"].unique() if t >= 0))
    hits = docs[docs["topic"].ge(0)]
    present = sparse.csr_matrix(
        (np.ones(len(hits)), (ids.get_indexer(hits["DFCI_MRN"]), np.searchsorted(topics, hits["topic"]))),
        shape=(len(ids), len(topics)))
    present = (present > 0).astype(np.int64).tocsc()
    keep = np.flatnonzero(np.asarray(present.sum(axis=0)).ravel() >= TERM_MIN_PATIENTS)
    if not keep.size:
        return pd.DataFrame(columns=TOPIC_COLS)
    res = fisher_enrichment(present[:, keep].tocsr(), groups.reindex(ids).to_numpy(), topics[keep], landmark,
                            strata_by_lm[landmark].reindex(ids).to_numpy())
    if res.empty:
        return pd.DataFrame(columns=TOPIC_COLS)
    res = res.rename(columns={"feature": "topic", "group_with_feature": "group_with_topic",
                              "control_with_feature": "control_with_topic"})
    return res.assign(top_words=res["topic"].map(topic_table.set_index("topic")["top_words"]))


if topic_docs is None:
    print("BERTopic section skipped; see the previous cell.")
else:
    topic_frames, coverage_rows = [], []
    keys = topic_docs.set_index("rpt_key")
    for lm in LANDMARKS:
        if lm not in reports_by_lm:
            continue
        grp = groups_by_lm[lm]
        in_window = rpt_key(reports_by_lm[lm]["RPT_ID"]).dropna().drop_duplicates()
        docs_lm = keys.loc[keys.index.intersection(pd.Index(in_window)), ["DFCI_MRN", "topic"]].reset_index()
        topic_frames.append(topic_enrichment(docs_lm, grp, lm))
        for name in (*TERM_GROUPS, CONTROL_GROUP):
            members = grp.index[grp.eq(name).to_numpy()]
            in_group = docs_lm["DFCI_MRN"].isin(members)
            coverage_rows.append({"landmark": lm, "group": name,
                                  "patients_with_report": group_sizes.loc[(lm, name), "patients_with_report"],
                                  "patients_with_embedded_report": int(docs_lm.loc[in_group, "DFCI_MRN"].nunique()),
                                  "embedded_reports": int(in_group.sum()),
                                  "outlier_reports": int((in_group & docs_lm["topic"].eq(-1)).sum())})
    topic_enrich = sort_contrasts(pd.concat(topic_frames, ignore_index=True).reindex(columns=TOPIC_COLS))
    topic_enrich_path = OUTPUT_DIR / "pathology_topic_enrichment.csv"
    topic_enrich.to_csv(topic_enrich_path, index=False)
    print(f"wrote {topic_enrich_path} ({len(topic_enrich):,} topic x contrast rows)")

    topic_coverage = pd.DataFrame(coverage_rows).set_index(["landmark", "group"])
    display(topic_coverage)
    topic_show = ["topic", "top_words", "group_with_topic", "n_group", "pct_group", "control_with_topic",
                  "n_control", "pct_control", "odds_ratio", "q_value", "odds_ratio_mh", "q_value_mh"]
    for lm in LANDMARKS:
        for name, ref in CONTRASTS:
            sub = topic_enrich[(topic_enrich["landmark"] == lm) & (topic_enrich["group"] == name)
                               & (topic_enrich["control"] == ref)]
            title = f"{contrast_title(lm, name, ref)} (topics)"
            if sub.empty:
                print(f"\n{title}: no topics (too few patients or reports)")
                continue
            print(f"\n{title} (q < {TERM_FDR}: {int((sub['q_value'] < TERM_FDR).sum()):,} topics)")
            display(sub[topic_show].head(TERM_TOP_N).round(3).reset_index(drop=True))

## Sensitivity and complementary analyses

Same groups, landmarks and pre-landmark reports as the term enrichment. Each one addresses a specific way the term
tables can mislead.

1. **Report count and specimen type (`*_mh` columns).** Patients with more reports have more chances to mention any
   term, and metastatic-site biopsies read differently from prostate specimens. Each patient's stratum is their
   in-window report-count band (`ADJUST_REPORT_BANDS`) crossed with whether any in-window report is from a
   non-prostate site (from `PROC_DESC` and the report's first `SPECIMEN_HEADER_CHARS` characters). The term, topic
   and lexicon tables add a one-sided, continuity-corrected Mantel-Haenszel test and MH odds ratio across strata.
   These are NaN when no stratum contains patients from both sides of the contrast.
2. **Negation (`neg_` terms).** With `TERM_NEGATION`, words inside a NegEx-style scope ("negative for synaptophysin,
   chromogranin and CD56", "carcinoma not identified") are counted as separate `neg_` terms. "Negative for small
   cell" therefore no longer counts as a "small cell" mention. The scope reaches `NEG_WINDOW` content words forward from
   a cue (half that backward from a post-cue). It stops at a sentence end, a line break, "but" / "however" or "positive".
   Pseudo-negations such as "cannot rule out" are ignored.
3. **Curated lexicon (`pathology_lexicon_enrichment.csv`).** The neuroendocrine patterns are copied from the NEPC
   annotation pipeline's `ne_gate.py`, joined by individual NE and prostate IHC markers, histologic variants, invasion
   / extension terms and the highest Gleason sum (grade group mapped to a sum). Mentions and negated mentions are
   separate features. Same Fisher and MH tests.
4. **Continuous risk (`pathology_score_association.csv`).** Deciles discard most of the score. For every term and
   lexicon feature, a two-sided Mann-Whitney test of the risk percentile among all scored patients with in-window
   reports, feature present vs absent. `auc` = P(a patient with the feature ranks higher than one without). The van
   Elteren test (`*_adj`) stratifies it like item 1.
5. **Discrimination (`pathology_text_classifier.csv`).** Can the pre-landmark text tell the groups apart at all? This is
   a cross-validated (`TEXT_FOLDS`) logistic regression on each patient's TF-IDF (L1) and, when the precomputed
   embeddings are available, on their mean report embedding (L2), compared with `TEXT_PERMUTATIONS` label
   permutations. Vocabulary selection (minimum distinct-patient frequency) and IDF are fitted within each training
   fold, including each permutation; bigrams never cross report boundaries. A fold with no retained terms uses an
   intercept-only prediction. TF-IDF `n_features` and top terms describe a separate fit on all patients in that contrast.

All outputs are aggregate. No MRN or report text is printed or written by these cells.


In [ ]:
# Curated lexicon, matched on lowercased report text. The NE patterns are copied verbatim (not imported) from
# LLM_clinical_annotations/tasks/nepc/ne_gate.py (NE_GATE_VERSION "ne-gate-v3", commit
# edb1713ef454f6ece313fa675b5f9438937f831e); update them by hand if that file changes.
NE_TERM = (r"\b(?:neuro[\s-]?endocrine|t-?nepc|nepc|small[\s-]?cell|scpc|scnc|"
           r"oat[\s-]?cell)\b")
SMALL_CELL_TERM = r"\b(?:small[\s-]?cell|scpc|scnc|oat[\s-]?cell)\b"
NE_DIAGNOSIS = (r"\b(?:t-?nepc|nepc|scpc|scnc)\b|"
                r"\b(?:neuro[\s-]?endocrine|small[\s-]?cell|oat[\s-]?cell)"
                r"(?:\s+[\w/()-]+){0,4}?\s+"
                r"(?:carcinoma|cancer|tumou?r|neoplasm|malignancy|component|histology|"
                r"variant|subtype|type|disease|prostate)\b|"
                r"\b(?:carcinoma|cancer)\s+(?:with|of)\s+(?:\w+\s+){0,2}?"
                r"(?:neuro[\s-]?endocrine|small[\s-]?cell)\b")
NE_MARKER = (r"\b(?:synaptophysin|chromogranin(?:\s+a)?|cd\s?56|insm-?1|"
             r"neuron[\s-]+specific\s+enolase|nse)\b")
NON_PROSTATE_NE = (r"\b(?:sclc|small[\s-]?cell\s+lung|merkel|carcinoid|"
                   r"pulmonary\s+(?:origin|primary|small)|lung\s+(?:origin|primary)|"
                   r"urothelial|bladder\s+(?:primary|origin|cancer|carcinoma|tumou?r)|"
                   r"pancreatic\s+(?:net|neuroendocrine)|pnet|gep[\s-]?nets?|"
                   r"well[\s-]differentiated\s+neuroendocrine\s+tumou?r)\b")
NON_SMALL_CELL = r"\bnon[\s-]*$"
# (end of the ne_gate.py copy)

LEXICON = {
    # neuroendocrine (ne_gate.py), then each NE marker from NE_MARKER on its own
    "NE term (any)": NE_TERM,
    "NE diagnosis phrase": NE_DIAGNOSIS,
    "small cell": SMALL_CELL_TERM,
    "NE marker (any)": NE_MARKER,
    "non-prostate NE primary": NON_PROSTATE_NE,
    "synaptophysin": r"\bsynaptophysin\b",
    "chromogranin": r"\bchromogranin\b",
    "CD56": r"\bcd\s?56\b",
    "INSM1": r"\binsm-?1\b",
    "NSE": r"\b(?:neuron[\s-]+specific\s+enolase|nse)\b",
    # other IHC
    "Ki-67 / MIB-1": r"\b(?:ki-?67|mib-?1)\b",
    "AR": r"\b(?:androgen\s+receptor|ar)\b",
    "NKX3.1": r"\bnkx\s?3[.\s-]?1\b",
    "PSMA": r"\b(?:psma|prostate[\s-]specific\s+membrane\s+antigen)\b",
    "ERG": r"\berg\b",
    "PTEN": r"\bpten\b",
    # histology and staging
    "ductal": r"\bductal\b",
    "intraductal": r"\bintraductal\b|\bidc-?p\b",
    "cribriform": r"\bcribriform\b",
    "sarcomatoid": r"\bsarcomatoid\b",
    "treatment effect": r"\b(?:treatment|therapy|hormon\w*|androgen[\s-]deprivation)[\s-]+(?:related\s+)?effects?\b",
    "perineural invasion": r"\bperineural\s+(?:invasion|infiltration)\b|\bpni\b",
    "lymphovascular invasion": (r"\b(?:lympho-?vascular|angiolymphatic|lymphatic|vascular)\s+(?:space\s+)?invasion\b"
                                r"|\blvi\b"),
    "extraprostatic extension": r"\bextra-?prostatic\s+extension\b|\bepe\b",
    "seminal vesicle invasion": r"\bseminal\s+vesicles?\s+(?:invasion|involvement|involved)\b|\bsvi\b",
}
LEXICON_RE = {name: re.compile(rx) for name, rx in LEXICON.items()}
NON_PREFIX_RE = re.compile(NON_SMALL_CELL)
# Gleason: the highest primary + secondary sum in any in-window report (grade group mapped to a sum).
GLEASON_RE = re.compile(r"\bgleason\b[^.;\n]{0,40}?\b([1-5])\s*\+\s*([1-5])\b")
GRADE_GROUP_RE = re.compile(r"\b(?:grade\s+group|isup\s+grade(?:\s+group)?|gg)\s*:?\s*([1-5])\b")
GRADE_GROUP_SUM = {1: 6, 2: 7, 3: 7, 4: 8, 5: 9}
GLEASON_BINS = (("Gleason max 2-6", 2, 6), ("Gleason max 7", 7, 7), ("Gleason max 8-10", 8, 10))

MENTION_FEATURES = [label for name in LEXICON for label in ((name, f"{name} (negated)") if TERM_NEGATION else (name,))]
LEXICON_FEATURES = MENTION_FEATURES + [label for label, _, _ in GLEASON_BINS]
LEXICON_COLS = ["landmark", "group", "control", "feature", "n_group", "n_control", "group_with_feature",
                "control_with_feature", "pct_group", "pct_control", "odds_ratio", "p_value", "q_value", *MH_COLS]


def lexicon_hits(text):
    """One report's lexicon features: {feature: True}, a negated mention as '<feature> (negated)', plus gleason_max.

    A mention is negated when any of its words is in a negation scope (see negated_spans); a match right after
    "non-" ("non-small cell") is not a mention.
    """
    text = text.lower()
    spans = negated_spans(text) if TERM_NEGATION else []
    starts = [s for s, _ in spans]
    hits = {}
    for name, rx in LEXICON_RE.items():
        for m in rx.finditer(text):
            if NON_PREFIX_RE.search(text[max(0, m.start() - 8):m.start()]):
                continue
            k = bisect.bisect_left(starts, m.end())       # spans[k - 1] is the last one starting inside the match
            hits[f"{name} (negated)" if k and spans[k - 1][1] > m.start() else name] = True
    sums = ([int(a) + int(b) for a, b in GLEASON_RE.findall(text)]
            + [GRADE_GROUP_SUM[int(g)] for g in GRADE_GROUP_RE.findall(text)])
    if sums:
        hits["gleason_max"] = max(sums)
    return hits


# Report-level hits, once for every in-window report at any landmark.
window_rows = np.unique(np.concatenate([rep["note_row"].to_numpy() for rep in reports_by_lm.values()]
                                       + [np.array([], dtype=int)])).astype(int)
report_lex = pd.DataFrame([lexicon_hits(t) for t in all_notes["RPT_TEXT"].to_numpy()[window_rows]],
                          index=window_rows).reindex(columns=MENTION_FEATURES + ["gleason_max"])
report_mentions = report_lex[MENTION_FEATURES].notna().to_numpy()
report_gleason = pd.to_numeric(report_lex["gleason_max"], errors="coerce").to_numpy(dtype=float)
lexicon_matrices = {}   # landmark -> (patient ids, presence); reused by the score association cell
lexicon_frames = []
for lm in LANDMARKS:
    if lm not in reports_by_lm or reports_by_lm[lm].empty:
        continue
    rep = reports_by_lm[lm]
    ids = pd.Index(sorted(rep["DFCI_MRN"].unique()), dtype="Int64")
    rows = ids.get_indexer(rep["DFCI_MRN"])
    pos = np.searchsorted(window_rows, rep["note_row"].to_numpy())
    mentions = np.zeros((len(ids), len(MENTION_FEATURES)), dtype=bool)
    np.logical_or.at(mentions, rows, report_mentions[pos])
    gleason = np.full(len(ids), np.nan)
    np.fmax.at(gleason, rows, report_gleason[pos])
    bins = np.column_stack([(gleason >= lo) & (gleason <= hi) for _, lo, hi in GLEASON_BINS])
    present = sparse.csr_matrix(np.hstack([mentions, bins]).astype(np.int64))
    lexicon_matrices[lm] = (ids, present)
    # Only features some patient has at this landmark are tested, so absent ones do not dilute the BH correction.
    seen = np.flatnonzero(colsum(present) > 0)
    lexicon_frames.append(fisher_enrichment(present[:, seen], groups_by_lm[lm].reindex(ids).to_numpy(),
                                            np.array(LEXICON_FEATURES, dtype=object)[seen], lm,
                                            strata_by_lm[lm].reindex(ids).to_numpy()))

# Rows stay in landmark, CONTRASTS, then LEXICON_FEATURES order.
lexicon_enrich = (pd.concat(lexicon_frames, ignore_index=True) if lexicon_frames else pd.DataFrame()
                  ).reindex(columns=LEXICON_COLS)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
lexicon_path = OUTPUT_DIR / "pathology_lexicon_enrichment.csv"
lexicon_enrich.to_csv(lexicon_path, index=False)
print(f"wrote {lexicon_path} ({len(lexicon_enrich):,} feature x contrast rows)")

# One wide table per landmark: % of patients with each feature by group, then q values for every contrast
# (q = Fisher, q_mh = stratified Mantel-Haenszel). Features no patient has at that landmark are not tested.
for lm in LANDMARKS:
    sub = lexicon_enrich[lexicon_enrich["landmark"] == lm]
    if sub.empty:
        print(f"\nLandmark {lm} · lexicon: no patients with in-window reports")
        continue
    wide = pd.DataFrame(index=pd.Index(LEXICON_FEATURES, name="feature"))
    for name, ref in CONTRASTS:
        c = sub[(sub["group"] == name) & (sub["control"] == ref)].set_index("feature")
        if c.empty:
            continue
        wide[f"% {name}"], wide[f"% {ref}"] = c["pct_group"] * 100, c["pct_control"] * 100
        wide[f"q {name}>{ref}"], wide[f"q_mh {name}>{ref}"] = c["q_value"], c["q_value_mh"]
    pct_cols = [f"% {g}" for g in (TERM_GROUPS[0], CONTROL_GROUP, TERM_GROUPS[1]) if f"% {g}" in wide]
    q_cols = [col for prefix in ("q ", "q_mh ") for col in wide.columns if col.startswith(prefix)]
    wide = wide[pct_cols + q_cols]
    print(f"\nLandmark {lm} · curated lexicon (% of patients with a pre-landmark mention)")
    display(wide.dropna(how="all").round(3))


In [ ]:
from scipy.stats import rankdata

ASSOC_COLS = ["landmark", "feature_set", "feature", "n_patients", "n_with_feature", "pct_with_feature",
              "mean_percentile_with", "mean_percentile_without", "auc", "p_value", "q_value",
              "auc_adj", "p_value_adj", "q_value_adj"]


def _mann_whitney_parts(present, score):
    """Per feature column: n with it, U of the with-feature patients, and the tie-corrected null variance of U."""
    n = present.shape[0]
    ranks = rankdata(score)
    n1 = colsum(present)
    u = np.asarray(present.T @ ranks).ravel() - n1 * (n1 + 1) / 2
    _, t = np.unique(score, return_counts=True)
    ties = float((t.astype(float) ** 3 - t).sum()) / (n * (n - 1)) if n > 1 else 0.0
    var = n1 * (n - n1) / 12 * ((n + 1) - ties)
    return n1, u, var


def score_association(present, features, score, strata, landmark, feature_set):
    """Two-sided Mann-Whitney of the risk percentile, feature present vs absent, among all scored patients with
    in-window reports; and the van Elteren test stratified like the Mantel-Haenszel columns.

    auc = P(percentile of a patient with the feature > one without); auc_adj pools the same comparison within strata.
    """
    present = present.tocsr().astype(float)
    n = present.shape[0]
    n1, u, var = _mann_whitney_parts(present, score)
    n0 = n - n1
    with np.errstate(divide="ignore", invalid="ignore"):
        z = np.maximum(np.abs(u - n1 * n0 / 2) - 0.5, 0) / np.sqrt(var)
        p = np.where(var > 0, 2 * norm.sf(z), np.nan)
        sum_with = np.asarray(present.T @ score).ravel()
        res = pd.DataFrame({
            "landmark": landmark, "feature_set": feature_set, "feature": features, "n_patients": n,
            "n_with_feature": n1.astype(int), "pct_with_feature": n1 / n,
            "mean_percentile_with": sum_with / n1, "mean_percentile_without": (score.sum() - sum_with) / n0,
            "auc": u / (n1 * n0), "p_value": p})
    # van Elteren: within-stratum U, strata weighted 1 / (N_k + 1).
    num, den, u_pool, pairs = (np.zeros(len(features)) for _ in range(4))
    for k in pd.unique(strata):
        rows = np.flatnonzero(strata == k)
        if rows.size < 2:
            continue
        n1k, uk, vk = _mann_whitney_parts(present[rows], score[rows])
        w = 1 / (rows.size + 1)
        num += w * (uk - n1k * (rows.size - n1k) / 2)
        den += w ** 2 * vk
        u_pool += uk
        pairs += n1k * (rows.size - n1k)
    with np.errstate(divide="ignore", invalid="ignore"):
        res["auc_adj"] = np.where(pairs > 0, u_pool / pairs, np.nan)
        res["p_value_adj"] = np.where(den > 0, 2 * norm.sf(np.abs(num) / np.sqrt(den)), np.nan)
    res = res[(res["n_with_feature"] > 0) & (res["n_with_feature"] < n)].reset_index(drop=True)
    res["q_value"], res["q_value_adj"] = bh_qvalues(res["p_value"]), bh_qvalues(res["p_value_adj"])
    return res[ASSOC_COLS]


assoc_frames = []
for lm in LANDMARKS:
    if lm not in by_lm:
        continue
    g = by_lm[lm]
    pct = pd.Series(g["risk_percentile"].to_numpy(dtype=float),
                    index=pd.Index(pd.to_numeric(g[fs.ID_COL], errors="coerce").astype("Int64")))
    sets = []
    if lm in term_matrices and len(term_matrices[lm][1]):
        ids, terms, present, _ = term_matrices[lm]
        sets.append(("term", ids, present, terms))
    if lm in lexicon_matrices:
        ids, present = lexicon_matrices[lm]
        sets.append(("lexicon", ids, present, np.array(LEXICON_FEATURES, dtype=object)))
    for feature_set, ids, present, names in sets:
        if len(ids) < 2:
            continue
        assoc_frames.append(score_association(present, names, pct.reindex(ids).to_numpy(dtype=float),
                                              strata_by_lm[lm].reindex(ids).to_numpy(), lm, feature_set))

assoc = (pd.concat(assoc_frames, ignore_index=True) if assoc_frames else pd.DataFrame()).reindex(columns=ASSOC_COLS)
assoc_path = OUTPUT_DIR / "pathology_score_association.csv"
assoc.to_csv(assoc_path, index=False)
print(f"wrote {assoc_path} ({len(assoc):,} feature x landmark rows)")

assoc_show = ["feature", "n_with_feature", "pct_with_feature", "mean_percentile_with", "mean_percentile_without",
              "auc", "q_value", "auc_adj", "q_value_adj"]
for lm in LANDMARKS:
    for feature_set in ("term", "lexicon"):
        sub = assoc[(assoc["landmark"] == lm) & (assoc["feature_set"] == feature_set)]
        if sub.empty:
            print(f"\nLandmark {lm} · {feature_set}s vs risk percentile: none tested")
            continue
        for direction, side in (("higher", sub["auc"] > 0.5), ("lower", sub["auc"] < 0.5)):
            hits = sub[side].sort_values("p_value")
            print(f"\nLandmark {lm} · {feature_set}s with {direction} risk percentiles "
                  f"(q < {TERM_FDR}: {int((hits['q_value'] < TERM_FDR).sum()):,}; top {TERM_TOP_N} by p)")
            display(hits[assoc_show].head(TERM_TOP_N).round(3).reset_index(drop=True))


In [ ]:
import sklearn
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, permutation_test_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

CLASSIFIER_PAIRS = (("top", CONTROL_GROUP), ("bottom", CONTROL_GROUP), ("top", "bottom"))   # (positive, negative)
CLASSIFIER_COLS = ["landmark", "positive_group", "negative_group", "features", "n_positive", "n_negative",
                   "n_features", "cv_auc", "null_auc_mean", "null_auc_95", "p_permutation",
                   "top_terms_positive", "top_terms_negative", "note"]
CLASSIFIER_TOP_TERMS = 10


class PatientTfidf(TransformerMixin, BaseEstimator):
    """Fit vocabulary and IDF on training patients, keeping each patient's report boundaries intact."""

    def __init__(self, min_patients=5, ngram_range=(1, 2), token_pattern=TOKEN_PATTERN):
        self.min_patients = min_patients
        self.ngram_range = ngram_range
        self.token_pattern = token_pattern

    def _terms(self, reports):
        for report in reports:
            yield from self.analyzer_(report)

    def fit(self, X, y=None):
        self.analyzer_ = CountVectorizer(lowercase=True, stop_words="english", token_pattern=self.token_pattern,
                                         ngram_range=self.ngram_range).build_analyzer()
        # Each input document is one patient's list of reports: min_df counts patients, not reports.
        self.vectorizer_ = CountVectorizer(analyzer=self._terms, min_df=self.min_patients)
        self.empty_ = len(X) < self.min_patients
        if not self.empty_:
            try:
                counts = self.vectorizer_.fit_transform(X)
            except ValueError as exc:
                if not any(message in str(exc) for message in ("empty vocabulary", "After pruning, no terms remain")):
                    raise
                self.empty_ = True
        if self.empty_:
            # A constant zero column lets logistic regression fit an intercept when a fold has no usable terms.
            self.names_ = np.array([""], dtype=object)
        else:
            self.names_ = self.vectorizer_.get_feature_names_out()
            self.tfidf_ = TfidfTransformer(sublinear_tf=True).fit(counts)
        return self

    def transform(self, X):
        if self.empty_:
            return sparse.csr_matrix((len(X), 1), dtype=float)
        return self.tfidf_.transform(self.vectorizer_.transform(X))

    def get_feature_names_out(self, input_features=None):
        return self.names_.copy()


def text_model(features):
    """L1 logistic on TF-IDF (sparse, so the kept terms are readable); L2 logistic on standardized embeddings."""
    if features == "embedding":
        return make_pipeline(StandardScaler(), LogisticRegression(C=TEXT_C, class_weight="balanced", max_iter=2000))
    kw = dict(solver="liblinear", C=TEXT_C, class_weight="balanced", max_iter=1000)
    # scikit-learn 1.8 deprecated penalty= in favour of l1_ratio=; older releases (the cluster) ignore l1_ratio here.
    if tuple(int(x) for x in sklearn.__version__.split(".")[:2]) >= (1, 8):
        classifier = LogisticRegression(l1_ratio=1.0, **kw)
    else:
        classifier = LogisticRegression(penalty="l1", **kw)
    return make_pipeline(PatientTfidf(min_patients=TERM_MIN_PATIENTS, ngram_range=TERM_NGRAMS), classifier)


def patient_embeddings(lm):
    """Mean precomputed embedding over each patient's in-window embedded reports at landmark lm."""
    if report_embeddings is None:
        return None, None
    keys = rpt_key(reports_by_lm[lm]["RPT_ID"]).dropna().unique()
    pos = embed_pos.reindex(keys).dropna().astype(int).to_numpy()
    if not pos.size:
        return None, None
    mrn = embed_docs["DFCI_MRN"].to_numpy()[pos]
    ids = pd.Index(sorted(set(mrn)), dtype="Int64")
    rows = ids.get_indexer(mrn)
    to_patient = sparse.csr_matrix((np.ones(len(rows)), (rows, np.arange(len(rows)))), shape=(len(ids), len(rows)))
    to_patient = sparse.diags(1 / np.asarray(to_patient.sum(axis=1)).ravel()) @ to_patient
    return ids, np.asarray(to_patient @ report_embeddings[pos])


def classify(X, y, features):
    model = text_model(features)
    cv = StratifiedKFold(TEXT_FOLDS, shuffle=True, random_state=TEXT_SEED)
    auc, null, p = permutation_test_score(model, X, y, cv=cv, n_permutations=TEXT_PERMUTATIONS, scoring="roc_auc",
                                          n_jobs=N_JOBS, random_state=TEXT_SEED)
    row = {"cv_auc": auc, "null_auc_mean": float(np.mean(null)), "null_auc_95": float(np.quantile(null, 0.95)),
           "p_permutation": p}
    if features == "tfidf":   # descriptive fit on all contrast patients, separate from the CV fits
        model.fit(X, y)
        names = model.steps[0][1].get_feature_names_out()
        row["n_features"] = int(np.count_nonzero(names != ""))
        coef = model.steps[-1][1].coef_.ravel()
        order = np.argsort(coef)
        row["top_terms_positive"] = ", ".join(names[i] for i in order[::-1][:CLASSIFIER_TOP_TERMS] if names[i] and coef[i] > 0)
        row["top_terms_negative"] = ", ".join(names[i] for i in order[:CLASSIFIER_TOP_TERMS] if names[i] and coef[i] < 0)
    return row


embed_pos = (pd.Series(np.arange(len(embed_docs)), index=embed_docs["rpt_key"].to_numpy())
             if embed_docs is not None else None)
if report_embeddings is None:
    print(f"Embedding classifier skipped ({embed_skip}); TF-IDF only.")

clf_rows = []
for lm in LANDMARKS:
    if lm not in reports_by_lm:
        continue
    grp = groups_by_lm[lm]
    feature_sets = {}
    reports = reports_by_lm[lm]
    if not reports.empty:
        pooled = reports.groupby("DFCI_MRN", sort=True)["TERM_TEXT"].agg(list)
        # A one-dimensional object array keeps whole patients together when sklearn indexes CV folds.
        documents = np.empty(len(pooled), dtype=object)
        documents[:] = pooled.tolist()
        feature_sets["tfidf"] = (pooled.index, documents)
    e_ids, e_x = patient_embeddings(lm)
    if e_ids is not None:
        feature_sets["embedding"] = (e_ids, e_x)
    for pos_name, neg_name in CLASSIFIER_PAIRS:
        for features, (f_ids, X) in feature_sets.items():
            label = grp.reindex(f_ids).to_numpy()
            keep = np.flatnonzero((label == pos_name) | (label == neg_name))
            y = (label[keep] == pos_name).astype(int)
            row = {"landmark": lm, "positive_group": pos_name, "negative_group": neg_name, "features": features,
                   "n_positive": int(y.sum()), "n_negative": int(len(y) - y.sum()), "n_features": X.shape[1] if features == "embedding" else np.nan}
            if min(row["n_positive"], row["n_negative"]) < TEXT_FOLDS:
                row["note"] = f"skipped: fewer than TEXT_FOLDS = {TEXT_FOLDS} patients in a group"
            else:
                row.update(classify(X[keep], y, features))
            clf_rows.append(row)

classifier = pd.DataFrame(clf_rows).reindex(columns=CLASSIFIER_COLS)
classifier_path = OUTPUT_DIR / "pathology_text_classifier.csv"
classifier.to_csv(classifier_path, index=False)
print(f"wrote {classifier_path} ({len(classifier):,} rows)")
print(f"\nCross-validated AUC ({TEXT_FOLDS}-fold) vs {TEXT_PERMUTATIONS} label permutations; "
      f"p resolution 1/{TEXT_PERMUTATIONS + 1}")
display(classifier.drop(columns=["top_terms_positive", "top_terms_negative"]).round(3))
for row in classifier.dropna(subset=["cv_auc"]).itertuples(index=False):
    if row.features == "tfidf":
        print(f"\nLandmark {row.landmark} · {row.positive_group} vs {row.negative_group} (TF-IDF, AUC {row.cv_auc:.2f})"
              f"\n  toward {row.positive_group}: {row.top_terms_positive or '(none)'}"
              f"\n  toward {row.negative_group}: {row.top_terms_negative or '(none)'}")
